In [9]:
!pip install diffusers transformers accelerate compel langchain 
# (plus any other Qwen/LangChain libraries you use)

In [11]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 81.1 MB/s eta 0:00:00:00:0100:01


In [6]:
!pip install -q langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.3 MB/s eta 0:00:00


In [7]:
!pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.1 MB/s eta 0:00:00:00:0100:01


In [19]:
import transformers
import torch
import langchain
import pydantic
import faiss
import json
import re
import gc
from typing import List, Optional, Any
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from langchain_core.language_models.llms import LLM
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain

# ==========================================
# 1. DEFINE PYDANTIC SCHEMA
# ==========================================
class MemoirChapter(BaseModel):
    title: str = Field(description="A poetic, evocative title for this memory")
    life_era: str = Field(description="Era of life: Childhood, Youth, Adulthood, or Later Years")
    estimated_year: Optional[int] = Field(description="Estimated four-digit year, or null if uncertain")
    location: Optional[str] = Field(description="Neighborhood, city, or setting mentioned")
    key_people: List[str] = Field(description="Names or roles of family members and friends mentioned")
    sensory_details: List[str] = Field(description="Sensory anchors: aromas, sounds, tastes, textures, weather")
    core_wisdom: str = Field(description="The underlying reflection, life lesson, or philosophical takeaway")
    narrative_text: str = Field(description="The memory rewritten into warm, literary detailed autobiography prose for the book ,Expand the narrative beautifully, making it at least 2-3 paragraphs long, exploring the emotions and atmosphere deeply.")
    illustration_prompt: str = Field(description="A highly detailed prompt for an AI image generator to create a warm watercolor or vintage pencil sketch illustrating this exact memory")

parser = PydanticOutputParser(pydantic_object=MemoirChapter)
format_instructions = parser.get_format_instructions()

# ==========================================
# 2. LOAD QWEN IN 4-BIT
# ==========================================
model_id = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

def run_qwen(prompt: str, max_new_tokens: int = 1200) -> str:
    messages = [
        {"role": "system", "content": "You are a thoughtful, literary biographer who listens to spoken human memories and crafts timeless memoirs while adhering strictly to requested output structures."},
        {"role": "user", "content": prompt}
    ]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

class CustomQwenLLM(LLM):
    def _call(self, prompt: str, stop: Any = None) -> str:
        return run_qwen(prompt)
    
    @property
    def _llm_type(self) -> str:
        return "custom_qwen"

llm = CustomQwenLLM()

# ==========================================
# 3. SET UP FAISS VECTOR DATABASE
# ==========================================
cultural_records = [
    Document(page_content="Mid-1960s: Communal home baking on Thursdays, aromas of fenugreek and fresh yeast, children running in neighborhood alleys before sunset."),
    Document(page_content="1950s-1960s: First Thursday of every month radio broadcasts of classical singers; families gathered around wooden tube radios."),
    Document(page_content="1970s: Alexandria waterfront, tramway commutes, coastal rainstorms, traditional coffee houses with tea and mint.")
]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectordb = FAISS.from_documents(cultural_records, embeddings)

# ==========================================
# 4. DEFINE EXTRACTION PIPELINE
# ==========================================
prompt_template = """Read the elder's spoken memory below. Structure the narrative into a polished memoir entry.

CRITICAL RULES:
1. The memory MUST take place exactly where the elder specifies. Do NOT change the location to match the historical context.
2. Only include sensory details that the elder actually mentioned (e.g., specific flowers, weather).
3. You may use the Cultural Context for subtle background tone, but the Elder's Spoken Memory is the absolute truth.
4. EXPAND THE NARRATIVE: Do not write a short summary. The `narrative_text` MUST be a lush, multi-paragraph story (at least 2 paragraphs) that deeply explores the mood, the environment, and the feelings described.

Cultural & Historical Context:
{context}

Spoken Memory:
"{spoken_input}"

Format Instructions:
{format_instructions}

Return ONLY valid JSON matching the format instructions above. Do not include markdown preamble.
"""

memoir_prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "spoken_input"],
    partial_variables={"format_instructions": format_instructions}
)

memoir_chain = LLMChain(llm=llm, prompt=memoir_prompt)

def generate_memoir_entry(user_story: str) -> MemoirChapter:
    matched_docs = vectordb.similarity_search(user_story, k=1)
    context_str = matched_docs[0].page_content if matched_docs else "Historical backdrop of mid-20th century daily life."
    
    response = memoir_chain.invoke({
        "context": context_str, 
        "spoken_input": user_story
    })
    raw_output = response["text"]
    
    match = re.search(r"```json\s*(.*?)\s*```", raw_output, re.DOTALL)
    clean_json = match.group(1).strip() if match else raw_output.strip()
    return parser.parse(clean_json)

print("✅ Qwen Model and Pipeline Loaded Successfully! Ready for FastAPI.")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Qwen Model and Pipeline Loaded Successfully! Ready for FastAPI.


In [17]:
!pip install -q pyngrok

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List, Dict
import uvicorn
from pyngrok import ngrok
import nest_asyncio

app = FastAPI()

class ChatRequest(BaseModel):
    messages: List[Dict[str, str]]

@app.post("/ask-biographer")
def ask_biographer(req: ChatRequest):
    # Build transcript from chat history
    full_transcript = "\n".join([f"{m['role'].capitalize()}: {m['content']}" for m in req.messages])
    
    interviewer_prompt = f"""You are a gentle, empathetic biographer interviewing an elderly person about their life.
Here is the conversation so far:
{full_transcript}
Based on what they just said, ask exactly ONE short, warm follow-up question. Try to encourage them to share a specific sensory detail (a smell, a sound, the weather, colors) or to name the people who were there. Do not write anything else except the question itself."""
    
    question = run_qwen(interviewer_prompt).strip()
    return {"question": question}

class FinalizeRequest(BaseModel):
    transcript: str

@app.post("/finalize")
def finalize(req: FinalizeRequest):
    chapter_pydantic = generate_memoir_entry(req.transcript)
    return chapter_pydantic.model_dump()

# ==========================================
# RUN THE SERVER 
# ==========================================

# 1. Clear any old sessions
ngrok.kill()

# 2. Authenticate
ngrok.set_auth_token("3IUyjbwmkFVVctGNigsUb4uxPl0_7NDEmk4gunMeZ2U64cCmT")

# 3. Connect WITHOUT the domain parameter to get a fresh random URL
public_url = ngrok.connect(8000).public_url
print(f"🚀 KAGGLE QWEN API URL: {public_url}")

# 4. Run the server safely
nest_asyncio.apply()
config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()

🚀 KAGGLE QWEN API URL: https://obstruct-eardrum-surely.ngrok-free.dev


INFO:     Started server process [188]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     154.182.61.166:0 - "POST /ask-biographer HTTP/1.1" 200 OK
INFO:     154.182.61.166:0 - "POST /ask-biographer HTTP/1.1" 200 OK
INFO:     154.182.61.166:0 - "POST /ask-biographer HTTP/1.1" 200 OK
INFO:     154.182.61.166:0 - "POST /ask-biographer HTTP/1.1" 200 OK
INFO:     154.182.61.166:0 - "POST /ask-biographer HTTP/1.1" 200 OK
INFO:     154.182.61.166:0 - "POST /finalize HTTP/1.1" 200 OK
INFO:     154.182.61.166:0 - "POST /ask-biographer HTTP/1.1" 200 OK
INFO:     154.182.61.166:0 - "POST /ask-biographer HTTP/1.1" 200 OK
INFO:     154.182.61.166:0 - "POST /ask-biographer HTTP/1.1" 200 OK
INFO:     154.182.61.166:0 - "POST /ask-biographer HTTP/1.1" 200 OK
INFO:     154.182.61.166:0 - "POST /ask-biographer HTTP/1.1" 200 OK
INFO:     154.182.61.166:0 - "POST /ask-biographer HTTP/1.1" 200 OK
INFO:     154.182.61.166:0 - "POST /ask-biographer HTTP/1.1" 200 OK
INFO:     154.182.61.166:0 - "POST /ask-biographer HTTP/1.1" 200 OK
INFO:     154.182.61.166:0 - "POST /ask-biographer HTT